# 06 — Heatmap de actividad (Fase 0)

**Objetivo:** Agregar centroides de tracks en grid → PNG + JSON (REQ-03 Analytics).


## Prerrequisitos

**03** → `tracks.jsonl`; metadata de **01** para dimensiones.


## 1. Setup


In [1]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt

from _common.io import (
    bbox_centroid,
    ensure_scripts_on_path,
    read_json,
    read_jsonl,
    setup_logging,
    stage_output_dir,
    utc_now_iso,
    write_json,
)
from loguru import logger

ensure_scripts_on_path()
setup_logging()


## 2. Configuration


In [2]:
TRACKS_PATH = stage_output_dir("03_track") / "tracks.jsonl"
CAPTURE_META = stage_output_dir("01_capture") / "metadata.json"
OUT_DIR = stage_output_dir("06_heatmap")
GRID_ROWS = 16
GRID_COLS = 24


## 3. Acumular grid


In [3]:
rows = read_jsonl(TRACKS_PATH)
frame_w, frame_h = 1920, 1080
if CAPTURE_META.is_file():
    m = read_json(CAPTURE_META)
    frame_w, frame_h = int(m["width"]), int(m["height"])

grid = np.zeros((GRID_ROWS, GRID_COLS), dtype=np.float32)
for r in rows:
    cx, cy = bbox_centroid(r["bbox"])
    col = min(GRID_COLS - 1, int(cx / max(frame_w, 1) * GRID_COLS))
    row = min(GRID_ROWS - 1, int(cy / max(frame_h, 1) * GRID_ROWS))
    grid[row, col] += 1.0

logger.info("Celdas activas: {}", int((grid > 0).sum()))


21:59:23 | INFO | Celdas activas: 8


## 4. Exportar PNG + JSON


In [4]:
heatmap_meta = {
    "grid_rows": GRID_ROWS,
    "grid_cols": GRID_COLS,
    "frame_width": frame_w,
    "frame_height": frame_h,
    "total_hits": float(grid.sum()),
    "created_at": utc_now_iso(),
    "values": grid.tolist(),
}
write_json(OUT_DIR / "heatmap.json", heatmap_meta)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(grid, origin="upper", cmap="hot", interpolation="nearest")
ax.set_title("VisionOps — heatmap de centroides")
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(OUT_DIR / "heatmap.png", dpi=120)
plt.close(fig)
print(f"OK — {OUT_DIR / 'heatmap.png'}")


OK — /Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/outputs/06_heatmap/heatmap.png


## 5. Validación


In [5]:
assert (OUT_DIR / "heatmap.png").is_file()
assert (OUT_DIR / "heatmap.json").is_file()
print("Heatmap generado correctamente")


Heatmap generado correctamente
